In [9]:
# ─── Install ──────────────────────────────────────────────────
!pip install -q transformers accelerate

import os, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# ─── Paths ────────────────────────────────────────────────────
PHASE1_IMAGES = '/content/drive/MyDrive/Train/Images'
PHASE1_MASKS  = '/content/drive/MyDrive/Train/Masks'

PHASE2_IMAGES = '/content/drive/MyDrive/GLOFdataset_2220'
PHASE2_MASKS  = '/content/drive/MyDrive/masks_2220'

ckpt_dir = '/content/drive/MyDrive/GLOF_two_phase'
p1_dir   = os.path.join(ckpt_dir, 'phase1')
p2_dir   = os.path.join(ckpt_dir, 'phase2')
os.makedirs(p1_dir, exist_ok=True)
os.makedirs(p2_dir, exist_ok=True)

p1_checkpoint = os.path.join(p1_dir, 'last_checkpoint.pt')
p1_best       = os.path.join(p1_dir, 'best_model')
p1_log        = os.path.join(p1_dir, 'training_log.csv')
p1_metrics    = os.path.join(p1_dir, 'phase1_metrics.json')

p2_checkpoint = os.path.join(p2_dir, 'last_checkpoint.pt')
p2_best       = os.path.join(p2_dir, 'best_model')
p2_log        = os.path.join(p2_dir, 'training_log.csv')
p2_metrics    = os.path.join(p2_dir, 'phase2_metrics.json')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ─── Dataset ──────────────────────────────────────────────────
class LakeDataset(Dataset):
    def __init__(self, file_list, img_dir, mask_dir, processor, augment=False, size=512):
        self.files, self.img_dir, self.mask_dir = file_list, img_dir, mask_dir
        self.processor, self.augment, self.size = processor, augment, size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img  = Image.open(os.path.join(self.img_dir,  fname)).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert('L')
        mask = mask.resize((self.size, self.size), Image.NEAREST)

        if self.augment:
            if np.random.rand() > 0.5:
                img, mask = img.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)
            if np.random.rand() > 0.5:
                img, mask = img.transpose(Image.FLIP_TOP_BOTTOM), mask.transpose(Image.FLIP_TOP_BOTTOM)
            if np.random.rand() > 0.5:
                angle = int(np.random.choice([90, 180, 270]))
                img, mask = img.rotate(angle), mask.rotate(angle)

        mask_np = (np.array(mask) > 127).astype(np.int64)
        enc = self.processor(images=img, return_tensors='pt')
        return enc['pixel_values'].squeeze(), torch.tensor(mask_np)

# ─── Loss ─────────────────────────────────────────────────────
ce_loss_fn = nn.CrossEntropyLoss()

def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.softmax(logits, dim=1)[:, 1, :, :]
    targets = targets.float()
    inter = (probs * targets).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + targets.sum(dim=(1, 2))
    return 1 - ((2*inter + smooth) / (union + smooth)).mean()

def combined_loss(logits, targets):
    return ce_loss_fn(logits, targets) + dice_loss(logits, targets)

# ─── Per-batch metrics (monitoring during training) ────────────
def compute_metrics(pred, target, smooth=1e-6):
    pred, target = (pred == 1).float(), (target == 1).float()
    tp = (pred * target).sum(); fp = (pred * (1-target)).sum()
    fn = ((1-pred) * target).sum(); tn = ((1-pred) * (1-target)).sum()
    dice = (2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou  = (tp+smooth)/(tp+fp+fn+smooth)
    prec = (tp+smooth)/(tp+fp+smooth)
    rec  = (tp+smooth)/(tp+fn+smooth)
    spec = (tn+smooth)/(tn+fp+smooth)
    acc  = (tp+tn+smooth)/(tp+tn+fp+fn+smooth)
    f1   = (2*prec*rec+smooth)/(prec+rec+smooth)
    return {'dice': dice.item(), 'iou': iou.item(), 'precision': prec.item(),
            'recall': rec.item(), 'specificity': spec.item(),
            'pixel_acc': acc.item(), 'f1': f1.item()}

# ─── Global metrics (final report, includes Cohen's Kappa) ─────
def cohens_kappa_from_counts(tp, fp, fn, tn, smooth=1e-6):
    n = tp+fp+fn+tn
    po = (tp+tn)/n
    pe = (((tp+fn)*(tp+fp)) + ((fp+tn)*(fn+tn))) / (n*n)
    return (po-pe+smooth)/(1-pe+smooth)

def compute_global_metrics(loader, model, device, smooth=1e-6):
    model.eval()
    tp_t = fp_t = fn_t = tn_t = 0.0
    with torch.no_grad():
        for pv, masks in loader:
            pv, masks = pv.to(device), masks.to(device)
            with autocast():
                out = model(pixel_values=pv)
                logits = nn.functional.interpolate(out.logits, size=masks.shape[-2:],
                                                     mode='bilinear', align_corners=False)
            preds = logits.argmax(dim=1)
            pf, tf = (preds==1).float(), (masks==1).float()
            tp_t += (pf*tf).sum().item(); fp_t += (pf*(1-tf)).sum().item()
            fn_t += ((1-pf)*tf).sum().item(); tn_t += ((1-pf)*(1-tf)).sum().item()

    dice = (2*tp_t+smooth)/(2*tp_t+fp_t+fn_t+smooth)
    iou  = (tp_t+smooth)/(tp_t+fp_t+fn_t+smooth)
    prec = (tp_t+smooth)/(tp_t+fp_t+smooth)
    rec  = (tp_t+smooth)/(tp_t+fn_t+smooth)
    spec = (tn_t+smooth)/(tn_t+fp_t+smooth)
    acc  = (tp_t+tn_t+smooth)/(tp_t+tn_t+fp_t+fn_t+smooth)
    f1   = (2*prec*rec+smooth)/(prec+rec+smooth)
    kappa = cohens_kappa_from_counts(tp_t, fp_t, fn_t, tn_t, smooth)
    return {'dice': dice, 'iou': iou, 'precision': prec, 'recall': rec,
            'specificity': spec, 'accuracy': acc, 'f1': f1, 'cohens_kappa': kappa}

# ─── Generic train/val loop with resume, used for BOTH phases ──
def run_training(model, train_loader, val_loader, optimizer, scheduler, scaler,
                  epochs, checkpoint_path, best_dir, log_path, patience=7):
    import csv
    start_epoch, best_dice, patience_counter = 0, 0.0, 0
    history = {'train_loss': [], 'val_loss': [], 'dice': [], 'iou': [],
               'pixel_acc': [], 'precision': [], 'recall': [], 'specificity': [], 'f1': []}

    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint...")
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        scaler.load_state_dict(ckpt['scaler_state'])
        start_epoch = ckpt['epoch'] + 1
        best_dice = ckpt['best_dice']
        patience_counter = ckpt['patience_counter']
        history = ckpt['history']
        print(f"Resumed at epoch {start_epoch}, best Dice so far: {best_dice:.4f}")
    else:
        print("Starting fresh training...")

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        for pv, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
            pv, masks = pv.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast():
                out = model(pixel_values=pv)
                logits = nn.functional.interpolate(out.logits, size=masks.shape[-2:],
                                                     mode='bilinear', align_corners=False)
                loss = combined_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss, totals = 0, None
        with torch.no_grad():
            for pv, masks in val_loader:
                pv, masks = pv.to(device), masks.to(device)
                with autocast():
                    out = model(pixel_values=pv)
                    logits = nn.functional.interpolate(out.logits, size=masks.shape[-2:],
                                                         mode='bilinear', align_corners=False)
                    loss = combined_loss(logits, masks)
                val_loss += loss.item()
                preds = logits.argmax(dim=1)
                m = compute_metrics(preds, masks)
                if totals is None: totals = {k: 0.0 for k in m}
                for k in m: totals[k] += m[k]

        n = len(val_loader)
        tl, vl = train_loss/len(train_loader), val_loss/n
        avg = {k: v/n for k, v in totals.items()}

        history['train_loss'].append(tl); history['val_loss'].append(vl)
        for k in ['dice','iou','pixel_acc','precision','recall','specificity','f1']:
            history[k].append(avg[k])

        print(f"Epoch {epoch+1:02d} | Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | "
              f"Dice: {avg['dice']:.4f} | IoU: {avg['iou']:.4f} | "
              f"Prec: {avg['precision']:.4f} | Rec: {avg['recall']:.4f} | F1: {avg['f1']:.4f}")

        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'scheduler_state': scheduler.state_dict(),
                    'scaler_state': scaler.state_dict(),
                    'best_dice': best_dice, 'patience_counter': patience_counter,
                    'history': history}, checkpoint_path)

        write_header = not os.path.exists(log_path)
        with open(log_path, 'a', newline='') as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow(['epoch','train_loss','val_loss','dice','iou',
                                  'pixel_acc','precision','recall','specificity','f1'])
            writer.writerow([epoch+1, tl, vl] + [avg[k] for k in
                             ['dice','iou','pixel_acc','precision','recall','specificity','f1']])

        if avg['dice'] > best_dice:
            best_dice, patience_counter = avg['dice'], 0
            model.save_pretrained(best_dir)
            processor.save_pretrained(best_dir)
            print(f"  Best model saved (Dice: {best_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    return history, best_dice

# ─── Processor (shared across both phases) ──────────────────────
processor = SegformerImageProcessor.from_pretrained(
    'nvidia/mit-b2', do_resize=True,
    size={'height': 512, 'width': 512}, do_normalize=True
)

# ════════════════════════════════════════════════════════════════
# PHASE 1 — Train on original 60 images (48 train / 12 val)
# ════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 1: Training on original 60 images")
print("="*60)

p1_files = sorted([f for f in os.listdir(PHASE1_IMAGES) if f.lower().endswith('.png')])
random.seed(42)
random.shuffle(p1_files)
split = int(0.8 * len(p1_files))   # 48/12
p1_train_files, p1_val_files = p1_files[:split], p1_files[split:]
print(f"Phase 1 — Train: {len(p1_train_files)} | Val: {len(p1_val_files)}")

if os.path.exists(p1_best):
    print("Phase 1 best model already exists — loading it, skipping training.")
    phase1_model = SegformerForSemanticSegmentation.from_pretrained(p1_best).to(device)
else:
    p1_train_ds = LakeDataset(p1_train_files, PHASE1_IMAGES, PHASE1_MASKS, processor, augment=True)
    p1_val_ds   = LakeDataset(p1_val_files,   PHASE1_IMAGES, PHASE1_MASKS, processor, augment=False)
    p1_train_loader = DataLoader(p1_train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
    p1_val_loader   = DataLoader(p1_val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

    phase1_model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/mit-b2', num_labels=2, ignore_mismatched_sizes=True
    ).to(device)

    p1_optimizer = torch.optim.AdamW(phase1_model.parameters(), lr=6e-5, weight_decay=0.01)
    P1_EPOCHS = 50
    p1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(p1_optimizer, T_max=P1_EPOCHS)
    p1_scaler = GradScaler()

    p1_history, p1_best_dice = run_training(
        phase1_model, p1_train_loader, p1_val_loader,
        p1_optimizer, p1_scheduler, p1_scaler,
        P1_EPOCHS, p1_checkpoint, p1_best, p1_log, patience=8
    )

    phase1_model = SegformerForSemanticSegmentation.from_pretrained(p1_best).to(device)

# Final Phase 1 metrics (global, includes Kappa)
p1_val_ds_final = LakeDataset(p1_val_files, PHASE1_IMAGES, PHASE1_MASKS, processor, augment=False)
p1_val_loader_final = DataLoader(p1_val_ds_final, batch_size=4, shuffle=False)
phase1_metrics = compute_global_metrics(p1_val_loader_final, phase1_model, device)

print("\n── Phase 1 Final Validation Metrics (12 images) ──")
for k, v in phase1_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(p1_metrics, 'w') as f:
    json.dump(phase1_metrics, f, indent=2)

# ════════════════════════════════════════════════════════════════
# PHASE 2 — Fine-tune on 2220 images (80/20 split)
# ════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 2: Fine-tuning on 2220 images")
print("="*60)

p2_img_names = sorted([f for f in os.listdir(PHASE2_IMAGES) if f.lower().endswith('.png')])
p2_msk_names = sorted([f for f in os.listdir(PHASE2_MASKS)  if f.lower().endswith('.png')])
p2_matched   = sorted(set(p2_img_names) & set(p2_msk_names))
print(f"Phase 2 matched pairs: {len(p2_matched)}")

p2_train_files, p2_val_files = train_test_split(p2_matched, test_size=0.20, random_state=42)
print(f"Phase 2 — Train: {len(p2_train_files)} | Val: {len(p2_val_files)}")

p2_train_ds = LakeDataset(p2_train_files, PHASE2_IMAGES, PHASE2_MASKS, processor, augment=True)
p2_val_ds   = LakeDataset(p2_val_files,   PHASE2_IMAGES, PHASE2_MASKS, processor, augment=False)
p2_train_loader = DataLoader(p2_train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
p2_val_loader   = DataLoader(p2_val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

if os.path.exists(p2_best):
    print("Phase 2 best model already exists — loading it, skipping training.")
    phase2_model = SegformerForSemanticSegmentation.from_pretrained(p2_best).to(device)
else:
    # Start fine-tuning from Phase 1's best model
    phase2_model = SegformerForSemanticSegmentation.from_pretrained(p1_best).to(device)

    p2_optimizer = torch.optim.AdamW(phase2_model.parameters(), lr=2e-5, weight_decay=0.01)  # lower LR for fine-tune
    P2_EPOCHS = 30
    p2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(p2_optimizer, T_max=P2_EPOCHS)
    p2_scaler = GradScaler()

    p2_history, p2_best_dice = run_training(
        phase2_model, p2_train_loader, p2_val_loader,
        p2_optimizer, p2_scheduler, p2_scaler,
        P2_EPOCHS, p2_checkpoint, p2_best, p2_log, patience=7
    )

    phase2_model = SegformerForSemanticSegmentation.from_pretrained(p2_best).to(device)

# Final Phase 2 metrics (global, includes Kappa)
p2_val_ds_final = LakeDataset(p2_val_files, PHASE2_IMAGES, PHASE2_MASKS, processor, augment=False)
p2_val_loader_final = DataLoader(p2_val_ds_final, batch_size=8, shuffle=False)
phase2_metrics = compute_global_metrics(p2_val_loader_final, phase2_model, device)

print("\n── Phase 2 Final Validation Metrics ──")
for k, v in phase2_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(p2_metrics, 'w') as f:
    json.dump(phase2_metrics, f, indent=2)

# ─── Summary ──────────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY — Phase 1 (60 imgs) vs Phase 2 (2220 imgs, fine-tuned)")
print("="*60)
print(f"{'Metric':<15} {'Phase 1':>10} {'Phase 2':>10}")
for k in ['dice','iou','precision','recall','specificity','accuracy','f1','cohens_kappa']:
    print(f"{k:<15} {phase1_metrics[k]:>10.4f} {phase2_metrics[k]:>10.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda

PHASE 1: Training on original 60 images
Phase 1 — Train: 48 | Val: 12
Phase 1 best model already exists — loading it, skipping training.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

/tmp/ipykernel_1879/4007494269.py:115: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



── Phase 1 Final Validation Metrics (12 images) ──
  dice: 0.5385
  iou: 0.3684
  precision: 0.7724
  recall: 0.4133
  specificity: 0.9962
  accuracy: 0.9787
  f1: 0.5385
  cohens_kappa: 0.5286

PHASE 2: Fine-tuning on 2220 images
Phase 2 matched pairs: 2220
Phase 2 — Train: 1776 | Val: 444


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

...

Epoch 28 | Train Loss: 0.1620 | Val Loss: 0.2148 | Dice: 0.8864 | IoU: 0.8062 | Prec: 0.8917 | Rec: 0.8905 | F1: 0.8864


Epoch 29 | Train Loss: 0.1635 | Val Loss: 0.2146 | Dice: 0.8863 | IoU: 0.8061 | Prec: 0.8930 | Rec: 0.8892 | F1: 0.8863


Epoch 30 | Train Loss: 0.1599 | Val Loss: 0.2147 | Dice: 0.8860 | IoU: 0.8055 | Prec: 0.8888 | Rec: 0.8926 | F1: 0.8860


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]


── Phase 2 Final Validation Metrics ──
  dice: 0.9085
  iou: 0.8324
  precision: 0.9079
  recall: 0.9092
  specificity: 0.9981
  accuracy: 0.9962
  f1: 0.9085
  cohens_kappa: 0.9066

SUMMARY — Phase 1 (60 imgs) vs Phase 2 (2220 imgs, fine-tuned)
Metric             Phase 1    Phase 2
dice                0.5385     0.9085
iou                 0.3684     0.8324
precision           0.7724     0.9079
recall              0.4133     0.9092
specificity         0.9962     0.9981
accuracy            0.9787     0.9962
f1                  0.5385     0.9085
cohens_kappa        0.5286     0.9066
